# 딥러닝및실습 Homework #2: Kaggle Titanic 생존자 예측

- 이름: 황명하
- 학번: 2022136136
- 대회: [Titanic - Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic)
- wandb 프로젝트: https://wandb.ai/audgk54587-dl/titanic

**목차**
1. 실행 환경
2. 데이터 준비와 전처리 이해
3. [요구사항 1] Titanic 딥러닝 모델 기본 훈련
4. [요구사항 2] Hyper Parameter 변경 및 선택
5. [요구사항 3] Early Stopping과 submission.csv 생성
6. [요구사항 4] Kaggle 제출 및 등수 확인
7. 종합 고찰
8. 숙제 후기

## 1. 실행 환경

모든 실험은 `torch.manual_seed(42)`로 시드를 고정했다. 따라서 같은 환경에서 같은 옵션으로 실행하면 train/validation 분할, 가중치 초기값, 결과 수치가 매번 동일하게 재현된다.

In [1]:
import sys, warnings
import torch, pandas as pd, sklearn, wandb

# titanic_dataset.py의 fillna(inplace=True) 문법 때문에 출력되는 pandas FutureWarning은 동작에 영향이 없으므로 숨김
warnings.filterwarnings("ignore", category=FutureWarning)

print("Python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("pandas :", pd.__version__)
print("sklearn:", sklearn.__version__)
print("wandb  :", wandb.__version__)

Python : 3.10.22
torch  : 2.14.1+cpu
pandas : 2.3.3
sklearn: 1.7.2
wandb  : 0.30.0


## 2. 데이터 준비와 전처리 이해

`titanic_dataset.py`는 `train.csv`(891명)와 `test.csv`(418명)를 합쳐서 아래 6단계 전처리를 수행한다.

| 함수 | 하는 일 |
|---|---|
| `get_preprocessed_dataset_1` | Fare 결측치를 같은 Pclass의 평균 요금으로 채움 |
| `get_preprocessed_dataset_2` | Name을 성(family_name), 호칭(title), 이름(name)으로 분리 |
| `get_preprocessed_dataset_3` | Age 결측치를 같은 title의 중앙값으로 채움 (Master는 어린 남자아이, Miss는 미혼 여성처럼 호칭마다 나이대가 다르기 때문) |
| `get_preprocessed_dataset_4` | family_num(=SibSp+Parch), alone 열을 추가하고 PassengerId, Name, Ticket, Cabin 등 삭제 |
| `get_preprocessed_dataset_5` | Mr/Miss/Mrs/Master 외의 드문 호칭은 other로 묶고, Embarked 결측치는 "missing"으로 채움 |
| `get_preprocessed_dataset_6` | 문자열 열(Sex, Embarked, title)을 LabelEncoder로 숫자로 변환 |

**코드를 직접 확인하며 발견한 점**
- **정규화(스케일링)를 하지 않는다.** Fare는 0~512, Age는 0~80인데 Pclass는 1~3, Sex는 0~1이라 feature마다 값의 범위 차이가 매우 크다. → 요구사항 1에서 표준화를 추가했고, 효과가 매우 컸다.
- **title 인코딩은 슬라이드 설명(0=Mr, 1=Miss, …)과 다르다.** LabelEncoder는 알파벳 순으로 번호를 매기므로 실제로는 Master=0, Miss=1, Mr=2, Mrs=3, other=4이다. 아래 출력에서 Mr인 첫 번째 승객의 title이 2인 것으로 확인할 수 있다.
- **random_split에 시드가 없어** 실행할 때마다 train/validation 구성이 달라진다. → 훈련 코드에서 시드를 고정해 실험 간 공정한 비교가 가능하도록 했다.
- Embarked(C/Q/S)는 순서가 없는 범주인데 0, 1, 2 숫자로 바뀌어서, 모델이 값 사이에 크기 관계가 있다고 오해할 여지가 있다. 원-핫 인코딩이 더 적절할 수 있다.

아래는 `titanic_dataset.py`를 실행한 결과이다. 전처리 후 feature는 10개이고 train 713개, validation 178개, test 418개로 나뉜다.

In [2]:
%run titanic_dataset.py

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'title', 'family_num', 'alone'],
      dtype='object')
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  title  \
0       0.0       3    1  22.0      1      0   7.2500         2      2   
1       1.0       1    0  38.0      1      0  71.2833         0      3   
2       1.0       3    0  26.0      0      0   7.9250         2      1   
3       1.0       1    0  35.0      1      0  53.1000         2      3   
4       0.0       3    1  35.0      0      0   8.0500         2      2   
5       0.0       3    1  29.0      0      0   8.4583         1      2   
6       0.0       1    1  54.0      0      0  51.8625         2      2   
7       0.0       3    1   2.0      3      1  21.0750         2      0   
8       1.0       3    0  27.0      0      2  11.1333         2      3   
9       1.0       2    0  14.0      1      0  30.0708         0      3   

   family_num  alone  
0           1    0.

## 3. [요구사항 1] Titanic 딥러닝 모델 기본 훈련

`_01_code/_08_learning_and_optimization/c_my_model_training_with_argparse_wandb.py`를 Titanic 데이터에 맞게 수정했다. 원본 대비 변경점은 코드에 `[변경]`, `[추가]` 주석으로 표시했다.

**주요 변경점**
- **데이터**: California Housing 대신 `get_preprocessed_dataset()` 사용. 배치가 튜플이 아니라 딕셔너리이므로 `batch['input']`, `batch['target']`으로 꺼냄
- **모델**: 입력 8개, 출력 1개 → **입력 10개, 출력 2개**(사망/생존 각각의 점수)
- **손실 함수**: 회귀용 `MSELoss` → 분류용 **`CrossEntropyLoss`** (target이 클래스 번호인 LongTensor이므로)
- **평가 지표**: validation accuracy 계산과 wandb 기록 추가
- **feature 표준화** 추가: 평균과 표준편차는 train 부분에서만 계산 (validation 정보 누출 방지)
- **하이퍼파라미터 인자화**: 활성화 함수, batch size, learning rate를 명령행 인자로 받음 (요구사항 2)
- **Early Stopping, 최적 가중치 복원, submission.csv 생성** (요구사항 3)

아래 셀은 코드를 노트북에 보여주면서 동시에 `titanic_training.py` 파일로 저장한다.

In [3]:
%%writefile titanic_training.py
import os
import copy
import torch
import pandas as pd
from torch import nn, optim
from torch.utils.data import DataLoader
from datetime import datetime
import wandb
import argparse

# [변경] California Housing 대신, 같은 폴더의 titanic_dataset.py에서 전처리된 데이터셋을 가져옴
from titanic_dataset import get_preprocessed_dataset

CURRENT_FILE_PATH = os.path.dirname(os.path.abspath(__file__))


def normalize_datasets(train_dataset, test_dataset):
  # [추가] feature 표준화: (x - 평균) / 표준편차
  # Fare(0~512), Age(0~80)와 Pclass(1~3)처럼 feature마다 값의 범위가 크게 달라서
  # 큰 값의 feature가 학습을 지배하거나 Sigmoid가 포화되는 문제를 막기 위함
  full_X = train_dataset.dataset.X              # train/validation이 공유하는 원본 텐서 (891 x 10)
  train_X = full_X[train_dataset.indices]       # 평균/표준편차는 train 부분에서만 계산 (validation 정보 누출 방지)
  mean = train_X.mean(dim=0)
  std = train_X.std(dim=0)
  std[std == 0] = 1.0                           # 0으로 나누기 방지

  train_dataset.dataset.X = (full_X - mean) / std   # train과 validation이 같은 텐서를 쓰므로 한 번에 변환됨
  test_dataset.X = (test_dataset.X - mean) / std    # test도 train 기준 평균/표준편차로 변환


def get_data(run):
  # [변경] train / validation / test 3개 데이터셋을 받아옴 (train/validation 분할은 titanic_dataset.py 내부에서 수행)
  train_dataset, validation_dataset, test_dataset = get_preprocessed_dataset()
  print("train: {0}, validation: {1}, test: {2}".format(
    len(train_dataset), len(validation_dataset), len(test_dataset)
  ))

  if run.config.normalize:
    normalize_datasets(train_dataset, test_dataset)

  train_data_loader = DataLoader(dataset=train_dataset, batch_size=run.config.batch_size, shuffle=True)
  validation_data_loader = DataLoader(dataset=validation_dataset, batch_size=len(validation_dataset))
  # [추가] submission.csv 생성용 test 데이터로더. test.csv의 PassengerId 순서와 맞아야 하므로 shuffle 하지 않음
  test_data_loader = DataLoader(dataset=test_dataset, batch_size=len(test_dataset))

  return train_data_loader, validation_data_loader, test_data_loader


# [추가] 요구사항 2에서 비교할 4가지 활성화 함수를 이름으로 선택할 수 있도록 사전으로 정의
ACTIVATIONS = {
  "sigmoid": nn.Sigmoid,
  "relu": nn.ReLU,
  "elu": nn.ELU,
  "leaky_relu": nn.LeakyReLU,
}


class MyModel(nn.Module):
  def __init__(self, n_input, n_output, run):
    super().__init__()

    # [변경] nn.ReLU 고정 대신, config에 지정된 활성화 함수를 사용
    activation = ACTIVATIONS[run.config.activation]

    self.model = nn.Sequential(
      nn.Linear(n_input, run.config.n_hidden_unit_list[0]),
      activation(),
      nn.Linear(run.config.n_hidden_unit_list[0], run.config.n_hidden_unit_list[1]),
      activation(),
      nn.Linear(run.config.n_hidden_unit_list[1], n_output),
      # 마지막에 Softmax를 넣지 않는 이유: nn.CrossEntropyLoss가 내부에서 softmax를 포함해 계산하기 때문
    )

  def forward(self, x):
    x = self.model(x)
    return x


def get_model_and_optimizer(run):
  # [변경] 입력 feature 10개, 출력 2개(0=사망, 1=생존 각각에 대한 점수)
  my_model = MyModel(n_input=10, n_output=2, run=run)
  # [변경] learning rate를 고정값 대신 config에서 받아옴 (요구사항 2)
  optimizer = optim.SGD(my_model.parameters(), lr=run.config.learning_rate)

  return my_model, optimizer


class EarlyStopping:
  # [추가] 요구사항 3: Early Stopping
  # - validation loss가 가장 낮았던 시점(best epoch)의 모델 가중치를 저장해 둠
  # - patience 동안 validation loss가 더 낮아지지 않으면 "더 학습해도 일반화 성능이 좋아지지 않는다(과적합 시작)"고 보고 훈련 중단
  def __init__(self, patience, min_delta=0.0):
    self.patience = patience
    self.min_delta = min_delta
    self.best_loss = float("inf")
    self.best_epoch = 0
    self.best_state = None
    self.counter = 0

  def step(self, validation_loss, model, epoch):
    if validation_loss < self.best_loss - self.min_delta:
      # 최고 기록 갱신: 이 시점의 가중치를 복사해서 보관 (deepcopy를 안 하면 이후 학습으로 값이 같이 바뀜)
      self.best_loss = validation_loss
      self.best_epoch = epoch
      self.best_state = copy.deepcopy(model.state_dict())
      self.counter = 0
    else:
      self.counter += 1

    return self.counter >= self.patience   # True면 훈련 중단


def evaluate(model, data_loader, loss_fn):
  # [추가] validation 데이터에 대한 loss와 정확도 계산 (훈련 중, 훈련 후 모두 사용)
  model.eval()
  loss_sum = 0.0
  num_batches = 0
  num_corrects = 0
  num_samples = 0
  with torch.no_grad():
    for batch in data_loader:
      input, target = batch['input'], batch['target']
      output = model(input)
      loss_sum += loss_fn(output, target).item()
      num_batches += 1

      # 두 출력 중 큰 쪽의 인덱스(0 또는 1)를 예측값으로 보고 정답과 비교
      predicted = output.argmax(dim=1)
      num_corrects += (predicted == target).sum().item()
      num_samples += len(target)

  return loss_sum / num_batches, num_corrects / num_samples


def training_loop(model, optimizer, train_data_loader, validation_data_loader, early_stopping, run):
  n_epochs = run.config.epochs
  # [변경] 회귀용 MSELoss -> 분류용 CrossEntropyLoss (target이 LongTensor인 클래스 번호이므로)
  loss_fn = nn.CrossEntropyLoss()
  next_print_epoch = 100

  for epoch in range(1, n_epochs + 1):
    model.train()
    loss_train = 0.0
    num_trains = 0
    for train_batch in train_data_loader:
      # [변경] titanic 데이터셋은 (input, target) 튜플이 아니라 딕셔너리를 반환
      input, target = train_batch['input'], train_batch['target']
      output_train = model(input)
      loss = loss_fn(output_train, target)
      loss_train += loss.item()
      num_trains += 1

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

    train_loss = loss_train / num_trains
    validation_loss, validation_accuracy = evaluate(model, validation_data_loader, loss_fn)

    run.log({
      "Epoch": epoch,
      "Training loss": train_loss,
      "Validation loss": validation_loss,
      "Validation accuracy": validation_accuracy,   # [추가]
    })

    if epoch >= next_print_epoch:
      print(
        f"Epoch {epoch}, "
        f"Training loss {train_loss:.4f}, "
        f"Validation loss {validation_loss:.4f}, "
        f"Validation accuracy {validation_accuracy:.4f}"
      )
      next_print_epoch += 100

    # [추가] 매 epoch마다 best 모델을 갱신하고, early stopping 조건을 확인
    should_stop = early_stopping.step(validation_loss, model, epoch)
    if run.config.early_stop and should_stop:
      print(
        f"Early stopping at epoch {epoch}: "
        f"{early_stopping.patience} epoch 동안 validation loss가 개선되지 않음 "
        f"(best epoch {early_stopping.best_epoch}, best validation loss {early_stopping.best_loss:.4f})"
      )
      break

  return loss_fn


def make_submission(model, test_data_loader, file_name):
  # [추가] 요구사항 3: test 데이터 예측 후 Kaggle 제출 형식(PassengerId, Survived)으로 저장
  model.eval()
  predictions = []
  with torch.no_grad():
    for test_batch in test_data_loader:
      output_test = model(test_batch['input'])
      predictions.append(output_test.argmax(dim=1))
  predictions = torch.cat(predictions).numpy()

  # 전처리 과정에서 PassengerId가 삭제되었으므로 원본 test.csv에서 가져옴 (행 순서는 동일하게 유지됨)
  test_df = pd.read_csv(os.path.join(CURRENT_FILE_PATH, "test.csv"))
  assert len(test_df) == len(predictions), "test.csv 행 수와 예측 개수가 다릅니다."

  submission_df = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": predictions.astype(int),
  })
  submission_path = os.path.join(CURRENT_FILE_PATH, file_name)
  submission_df.to_csv(submission_path, index=False)

  print("#" * 50, "submission")
  print(f"Saved: {submission_path} ({len(submission_df)} rows)")
  print(submission_df.head())
  print(f"예측 생존 비율: {submission_df['Survived'].mean():.4f}")


def main(args):
  current_time_str = datetime.now().astimezone().strftime('%Y-%m-%d_%H-%M-%S')

  # [추가] 시드 고정: titanic_dataset.py의 random_split도 이 시드를 따르므로
  # 실험마다 train/validation 분할이 같아져서 하이퍼파라미터를 공정하게 비교할 수 있음
  torch.manual_seed(args.seed)

  config = {
    'epochs': args.epochs,
    'batch_size': args.batch_size,
    'learning_rate': args.learning_rate,   # [변경] 1e-3 고정 -> 인자로 받음
    'activation': args.activation,         # [추가]
    'n_hidden_unit_list': [20, 20],
    'seed': args.seed,
    'normalize': args.normalize,           # [추가]
    'early_stop': args.early_stop,         # [추가]
    'patience': args.patience,             # [추가]
  }

  with wandb.init(
    mode="online" if args.wandb else "disabled",
    project="titanic",                                  # [변경]
    notes="Titanic survival classification",
    tags=["my_model", "titanic"],                       # [변경]
    # [변경] wandb에서 run을 구분하기 쉽도록 하이퍼파라미터를 이름에 포함
    name=f"{args.activation}_bs{args.batch_size}_lr{args.learning_rate}_{'norm' if args.normalize else 'raw'}{'_es' if args.early_stop else ''}_{current_time_str}",
    config=config
  ) as run:
    print(args)
    print(run.config)

    # 가로축을 Epoch, 세로축을 Training loss로 그림
    run.define_metric("Training loss", step_metric="Epoch")
    # 가로축을 Epoch, 세로축을 Validation loss로 그림, 실험 요약에는 가장 낮았던 값을 남김
    run.define_metric("Validation loss", step_metric="Epoch", summary="min")
    # [추가] 정확도는 가장 높았던 값을 요약에 남김
    run.define_metric("Validation accuracy", step_metric="Epoch", summary="max")

    train_data_loader, validation_data_loader, test_data_loader = get_data(run)

    my_model, optimizer = get_model_and_optimizer(run)
    early_stopping = EarlyStopping(patience=args.patience)

    print("#" * 50, 1)

    loss_fn = training_loop(
      model=my_model,
      optimizer=optimizer,
      train_data_loader=train_data_loader,
      validation_data_loader=validation_data_loader,
      early_stopping=early_stopping,
      run=run
    )

    # [추가] 마지막 epoch의 모델이 아니라, validation loss가 가장 낮았던 epoch의 가중치로 되돌림
    my_model.load_state_dict(early_stopping.best_state)
    best_validation_loss, best_validation_accuracy = evaluate(my_model, validation_data_loader, loss_fn)
    print("#" * 50, "best model")
    print(
      f"Best epoch {early_stopping.best_epoch}, "
      f"Validation loss {best_validation_loss:.4f}, "
      f"Validation accuracy {best_validation_accuracy:.4f}"
    )
    run.summary["Best epoch"] = early_stopping.best_epoch
    run.summary["Best validation accuracy"] = best_validation_accuracy

    # [추가] --submission 파일명을 지정한 경우에만, best 모델로 test 데이터를 예측해서 제출 파일 생성
    if args.submission:
      make_submission(my_model, test_data_loader, args.submission)


if __name__ == "__main__":
  parser = argparse.ArgumentParser()

  parser.add_argument(
    "--wandb", action=argparse.BooleanOptionalAction, default=True, help="True or False"
  )

  parser.add_argument(
    "-b", "--batch_size", type=int, default=32, help="Batch size (int, default: 32)"
  )

  parser.add_argument(
    "-e", "--epochs", type=int, default=1_000, help="Number of training epochs (int, default:1_000)"
  )

  # [추가] 요구사항 2용 인자
  parser.add_argument(
    "-l", "--learning_rate", type=float, default=1e-3, help="Learning rate (float, default: 1e-3)"
  )

  parser.add_argument(
    "-a", "--activation", type=str, default="relu", choices=list(ACTIVATIONS.keys()),
    help="Activation function (default: relu)"
  )

  parser.add_argument(
    "--seed", type=int, default=42, help="Random seed (int, default: 42)"
  )

  # [추가] feature 표준화 여부 (기본: 사용, --no-normalize로 끌 수 있음)
  parser.add_argument(
    "--normalize", action=argparse.BooleanOptionalAction, default=True, help="Standardize features (default: True)"
  )

  # [추가] 요구사항 3용 인자
  parser.add_argument(
    "--early_stop", action=argparse.BooleanOptionalAction, default=True, help="Use early stopping (default: True)"
  )

  parser.add_argument(
    "-p", "--patience", type=int, default=100, help="Early stopping patience in epochs (int, default: 100)"
  )

  parser.add_argument(
    "--submission", type=str, default=None, help="Submission file name, e.g. submission.csv (default: None = 생성 안 함)"
  )

  args = parser.parse_args()

  main(args)

Overwriting titanic_training.py


### 3-1. 정규화 없이 훈련 (ReLU, batch 32, lr 1e-3, 1000 epoch)

먼저 원본 데이터 그대로(`--no-normalize`) 훈련해 본다. 요구사항 1, 2에서는 1000 epoch 전체를 비교하기 위해 Early Stopping을 끈다(`--no-early_stop`).

In [4]:
%run titanic_training.py -a relu -b 32 -l 1e-3 --no-normalize --no-early_stop

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\audgk\_netrc.
wandb: Currently logged in as: audgk54587 (audgk54587-dl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Namespace(wandb=True, batch_size=32, epochs=1000, learning_rate=0.001, activation='relu', seed=42, normalize=False, early_stop=False, patience=100, submission=None)
{'epochs': 1000, 'batch_size': 32, 'learning_rate': 0.001, 'activation': 'relu', 'n_hidden_unit_list': [20, 20], 'seed': 42, 'normalize': False, 'early_stop': False, 'patience': 100}
Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'title', 'family_num', 'alone'],
      dtype='object')
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  title  \
0       0.0       3    1  22.0      1      0   7.2500         2      2   
1       1.0       1    0  38.0      1      0  71.2833         0      3   
2       1.0       3    0  26.0      0      0   7.9250         2      1   
3       1.0       1    0  35.0      1      0  53.1000         2      3   
4       0.0       3    1  35.0      0      0   8.0500         2      2   
5       0.0       3    1  29.0      0      0   8.4583         1  

Epoch,▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇██
Training loss,████▇▇▇▇▇▇▆▇▆▆▆▆▆▆▆▆▅▅▅▄▅▄▃▃▃▃▄▅▃▃▃▂▁▁▂▂
Validation accuracy,▁▂▂▂▁▃▃▃▂▂▂▂▂▃▁▁▂▃▂▂▄▂▂▂▃▄▂▄▄▇▄▅▄▇▇▅▅█▅▇
Validation loss,██▇▇▇▇█▇█▇██▆▆▇▆▇▇▆▆▆▅▆▇▆▆▅▇▆▅▆▅▄▅▅▄▄▄▆▁
Best epoch,995
Best validation accuracy,0.79775
Epoch,1000
Training loss,0.48619


### 3-2. 정규화 적용 후 훈련 (ReLU, batch 32, lr 1e-3, 1000 epoch)

In [5]:
%run titanic_training.py -a relu -b 32 -l 1e-3 --no-early_stop

Namespace(wandb=True, batch_size=32, epochs=1000, learning_rate=0.001, activation='relu', seed=42, normalize=True, early_stop=False, patience=100, submission=None)
{'epochs': 1000, 'batch_size': 32, 'learning_rate': 0.001, 'activation': 'relu', 'n_hidden_unit_list': [20, 20], 'seed': 42, 'normalize': True, 'early_stop': False, 'patience': 100}
Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'title', 'family_num', 'alone'],
      dtype='object')
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  title  \
0       0.0       3    1  22.0      1      0   7.2500         2      2   
1       1.0       1    0  38.0      1      0  71.2833         0      3   
2       1.0       3    0  26.0      0      0   7.9250         2      1   
3       1.0       1    0  35.0      1      0  53.1000         2      3   
4       0.0       3    1  35.0      0      0   8.0500         2      2   
5       0.0       3    1  29.0      0      0   8.4583         1    

Epoch,▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇██
Training loss,█▇▇▇▆▆▅▅▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation accuracy,▁▄▅▅▅▆▆▆▇▇▇▇▇███████████████████████████
Validation loss,█▇▇▇▇▆▅▅▅▅▄▄▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Best epoch,969
Best validation accuracy,0.88764
Epoch,1000
Training loss,0.40654


### 3-3. 결과 비교와 고찰

| 실험 | Epoch 1000 Training loss | Epoch 1000 Validation loss | Epoch 1000 Validation accuracy |
|---|---|---|---|
| 정규화 없음 | 0.4862 | 0.4641 | 0.7978 |
| 정규화 적용 | **0.4065** | **0.3612** | **0.8876** |

- 정규화를 하지 않으면 training loss가 1000 epoch 동안 0.58에서 0.49로 느리게 내려가고, validation loss도 중간에 0.52에서 0.58로 튀는 등 불안정했다.
- 정규화만 추가했는데 validation accuracy가 약 9%p 올랐고, loss도 훨씬 빠르고 매끄럽게 수렴했다.
- 이유: Fare처럼 값이 큰 feature는 가중치에 대한 기울기도 크게 만들어서 SGD 업데이트가 그 feature에 휘둘린다. 모든 feature를 평균 0, 표준편차 1로 맞추면 각 가중치가 비슷한 속도로 학습된다.
- Training loss와 Validation loss의 epoch별 변화 그래프는 wandb에 기록했고, 별도 제출 파일(`...-hw2-wandb.pdf`)에 포함했다.

## 4. [요구사항 2] Hyper Parameter 변경 및 선택

한 번에 하나의 하이퍼파라미터만 바꾸고 나머지는 고정하여 비교했다. 시드가 고정되어 있으므로 모든 실험은 같은 데이터 분할과 같은 초기 가중치에서 시작하며, 결과 차이는 바꾼 하이퍼파라미터 때문이라고 볼 수 있다. 모든 실험은 정규화를 적용하고 1000 epoch 동안 훈련했다.

판단 기준은 **validation loss**로 했다. validation 데이터가 178명뿐이라 accuracy는 한두 명의 예측만 바뀌어도 0.56%p씩 변하므로, 연속적인 값인 loss가 더 안정적인 지표이다.

### 4-1. Activation Function 비교 (batch 32, lr 1e-3 고정)

ReLU 결과는 3-2와 동일하므로 나머지 세 가지를 실행한다.

In [6]:
%run titanic_training.py -a sigmoid -b 32 -l 1e-3 --no-early_stop

Namespace(wandb=True, batch_size=32, epochs=1000, learning_rate=0.001, activation='sigmoid', seed=42, normalize=True, early_stop=False, patience=100, submission=None)
{'epochs': 1000, 'batch_size': 32, 'learning_rate': 0.001, 'activation': 'sigmoid', 'n_hidden_unit_list': [20, 20], 'seed': 42, 'normalize': True, 'early_stop': False, 'patience': 100}
Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'title', 'family_num', 'alone'],
      dtype='object')
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  title  \
0       0.0       3    1  22.0      1      0   7.2500         2      2   
1       1.0       1    0  38.0      1      0  71.2833         0      3   
2       1.0       3    0  26.0      0      0   7.9250         2      1   
3       1.0       1    0  35.0      1      0  53.1000         2      3   
4       0.0       3    1  35.0      0      0   8.0500         2      2   
5       0.0       3    1  29.0      0      0   8.4583        

Epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
Training loss,█▇▆▇▄▅▆▅▄▇▇▇▇▅▅▇▅▆▅▅█▄▆▅▅▃▅▆▆▄▃▆▃▄▂▂▂▁▁▂
Validation accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation loss,█▃▂▂▂▂▂▂▂▂▂▂▁▂▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Best epoch,1000
Best validation accuracy,0.70787
Epoch,1000
Training loss,0.66358


In [7]:
%run titanic_training.py -a elu -b 32 -l 1e-3 --no-early_stop

Namespace(wandb=True, batch_size=32, epochs=1000, learning_rate=0.001, activation='elu', seed=42, normalize=True, early_stop=False, patience=100, submission=None)
{'epochs': 1000, 'batch_size': 32, 'learning_rate': 0.001, 'activation': 'elu', 'n_hidden_unit_list': [20, 20], 'seed': 42, 'normalize': True, 'early_stop': False, 'patience': 100}
Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'title', 'family_num', 'alone'],
      dtype='object')
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  title  \
0       0.0       3    1  22.0      1      0   7.2500         2      2   
1       1.0       1    0  38.0      1      0  71.2833         0      3   
2       1.0       3    0  26.0      0      0   7.9250         2      1   
3       1.0       1    0  35.0      1      0  53.1000         2      3   
4       0.0       3    1  35.0      0      0   8.0500         2      2   
5       0.0       3    1  29.0      0      0   8.4583         1      

Epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
Training loss,█▆▂▂▂▂▂▂▂▁▂▂▂▂▂▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▂▁▁▁▁▁▂▁▁
Validation accuracy,▁▃▄▅▅▆▇▇▇███████████████████████████████
Validation loss,██▆▆▅▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Best epoch,982
Best validation accuracy,0.87079
Epoch,1000
Training loss,0.43527


In [8]:
%run titanic_training.py -a leaky_relu -b 32 -l 1e-3 --no-early_stop

Namespace(wandb=True, batch_size=32, epochs=1000, learning_rate=0.001, activation='leaky_relu', seed=42, normalize=True, early_stop=False, patience=100, submission=None)
{'epochs': 1000, 'batch_size': 32, 'learning_rate': 0.001, 'activation': 'leaky_relu', 'n_hidden_unit_list': [20, 20], 'seed': 42, 'normalize': True, 'early_stop': False, 'patience': 100}
Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'title', 'family_num', 'alone'],
      dtype='object')
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  title  \
0       0.0       3    1  22.0      1      0   7.2500         2      2   
1       1.0       1    0  38.0      1      0  71.2833         0      3   
2       1.0       3    0  26.0      0      0   7.9250         2      1   
3       1.0       1    0  35.0      1      0  53.1000         2      3   
4       0.0       3    1  35.0      0      0   8.0500         2      2   
5       0.0       3    1  29.0      0      0   8.4583  

Epoch,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇███
Training loss,██▇▇▇▇▆▆▅▅▅▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁
Validation accuracy,▁▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████████████████████████
Validation loss,█▇▇▇▆▆▆▆▆▅▅▅▅▅▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Best epoch,969
Best validation accuracy,0.88202
Epoch,1000
Training loss,0.40689


| 활성화 함수 | Epoch 200 Validation loss | Epoch 1000 Training loss | Epoch 1000 Validation loss | Epoch 1000 Validation accuracy |
|---|---|---|---|---|
| Sigmoid | 0.6315 | 0.6636 | 0.6239 | 0.7079 |
| ReLU | 0.4977 | 0.4065 | **0.3612** | **0.8876** |
| ELU | **0.4261** | 0.4353 | 0.3770 | 0.8708 |
| Leaky ReLU | 0.4971 | 0.4069 | 0.3613 | **0.8876** |

- **Sigmoid는 사실상 학습에 실패했다.** accuracy가 1000 epoch 내내 0.7079로 고정이고, training loss도 0.676에서 0.664로 거의 줄지 않았다. 거의 모든 승객을 "사망"으로 예측하는 상태이다. Sigmoid의 기울기는 최대 0.25라서 층을 지날수록 기울기가 작아지고(기울기 소실), 여기에 작은 learning rate의 SGD가 겹쳐 가중치가 거의 업데이트되지 않았다.
- **ReLU와 Leaky ReLU는 거의 같다**(0.3612 대 0.3613). 두 함수는 음수 입력에서만 다르고(0 대 0.01x), 같은 초기 가중치에서 시작했기 때문이다. 이 모델에서는 ReLU의 약점인 "죽은 뉴런" 문제가 거의 발생하지 않았다고 볼 수 있다.
- **ELU는 초반 수렴이 가장 빨랐지만**(epoch 200 validation loss 0.4261), 1000 epoch 후 최종 성능은 ReLU 계열보다 약간 낮았다.
- **선택: ReLU.** 가장 낮은 validation loss를 기록했고, Leaky ReLU와 사실상 동점이므로 더 단순한 쪽을 택했다.

### 4-2. Batch Size 비교 (ReLU, lr 1e-3 고정)

batch 32 결과는 3-2와 동일하므로 64와 128을 실행한다.

In [9]:
%run titanic_training.py -a relu -b 64 -l 1e-3 --no-early_stop

Namespace(wandb=True, batch_size=64, epochs=1000, learning_rate=0.001, activation='relu', seed=42, normalize=True, early_stop=False, patience=100, submission=None)
{'epochs': 1000, 'batch_size': 64, 'learning_rate': 0.001, 'activation': 'relu', 'n_hidden_unit_list': [20, 20], 'seed': 42, 'normalize': True, 'early_stop': False, 'patience': 100}
Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'title', 'family_num', 'alone'],
      dtype='object')
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  title  \
0       0.0       3    1  22.0      1      0   7.2500         2      2   
1       1.0       1    0  38.0      1      0  71.2833         0      3   
2       1.0       3    0  26.0      0      0   7.9250         2      1   
3       1.0       1    0  35.0      1      0  53.1000         2      3   
4       0.0       3    1  35.0      0      0   8.0500         2      2   
5       0.0       3    1  29.0      0      0   8.4583         1    

Epoch,▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇██
Training loss,██▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▃▃▂▂▂▂▂▂▁▁▁▁▂▁▂▁▁▂▂▁▂▂▂
Validation accuracy,▁▂▁▁▁▁▁▁▁▂▂▂▂▃▃▄▄▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇████████
Validation loss,██▇▇▇▆▆▆▆▅▅▅▄▄▄▄▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Best epoch,1000
Best validation accuracy,0.87079
Epoch,1000
Training loss,0.42786


In [10]:
%run titanic_training.py -a relu -b 128 -l 1e-3 --no-early_stop

Namespace(wandb=True, batch_size=128, epochs=1000, learning_rate=0.001, activation='relu', seed=42, normalize=True, early_stop=False, patience=100, submission=None)
{'epochs': 1000, 'batch_size': 128, 'learning_rate': 0.001, 'activation': 'relu', 'n_hidden_unit_list': [20, 20], 'seed': 42, 'normalize': True, 'early_stop': False, 'patience': 100}
Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'title', 'family_num', 'alone'],
      dtype='object')
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  title  \
0       0.0       3    1  22.0      1      0   7.2500         2      2   
1       1.0       1    0  38.0      1      0  71.2833         0      3   
2       1.0       3    0  26.0      0      0   7.9250         2      1   
3       1.0       1    0  35.0      1      0  53.1000         2      3   
4       0.0       3    1  35.0      0      0   8.0500         2      2   
5       0.0       3    1  29.0      0      0   8.4583         1  

Epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
Training loss,██▇▇▇▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁
Validation accuracy,▁▄▄▄▄▄▄▄▄▄▄▄▄▄▄▅▅▆▅▆▆▆▆▆▆▆▇▇▇▇▇▇████████
Validation loss,██▇▇▇▇▆▆▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁
Best epoch,1000
Best validation accuracy,0.85393
Epoch,1000
Training loss,0.47987


| Batch size | epoch당 업데이트 횟수 | 총 업데이트 횟수 | Epoch 200 Validation loss | Epoch 1000 Training loss | Epoch 1000 Validation loss | Epoch 1000 Validation accuracy |
|---|---|---|---|---|---|---|
| 32 | 23 | 23,000 | **0.4977** | **0.4065** | **0.3612** | **0.8876** |
| 64 | 12 | 12,000 | 0.5848 | 0.4279 | 0.3749 | 0.8708 |
| 128 | 6 | 6,000 | 0.6266 | 0.4799 | 0.4447 | 0.8539 |

- epoch 수를 고정하면 batch가 작을수록 성능이 좋았다. 핵심 원인은 **가중치 업데이트 횟수**로, batch 32는 128보다 같은 1000 epoch 동안 약 4배 많이 가중치를 갱신했다.
- batch 128은 epoch 900에서 1000 사이에도 validation loss가 0.4654에서 0.4447로 계속 빠르게 내려가고 있었다. 즉 batch 128이 나쁘다기보다 **같은 epoch 안에 수렴할 시간이 부족했던 것**이다. 업데이트 횟수를 맞추거나(epoch를 늘리거나) batch가 커진 만큼 learning rate를 키우면 결과가 달라질 수 있다.
- 작은 batch는 기울기 추정에 노이즈가 많지만 업데이트를 자주 하고, 이 노이즈가 일반화에 도움이 되기도 한다. 데이터가 713개로 작아서 batch 32도 속도 부담이 없었다.
- **선택: batch size 32.**

### 4-3. Learning Rate 비교 (ReLU, batch 32 고정)

lr 1e-3 결과는 3-2와 동일하므로 1e-2와 1e-4를 실행한다.

In [11]:
%run titanic_training.py -a relu -b 32 -l 1e-2 --no-early_stop

Namespace(wandb=True, batch_size=32, epochs=1000, learning_rate=0.01, activation='relu', seed=42, normalize=True, early_stop=False, patience=100, submission=None)
{'epochs': 1000, 'batch_size': 32, 'learning_rate': 0.01, 'activation': 'relu', 'n_hidden_unit_list': [20, 20], 'seed': 42, 'normalize': True, 'early_stop': False, 'patience': 100}
Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'title', 'family_num', 'alone'],
      dtype='object')
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  title  \
0       0.0       3    1  22.0      1      0   7.2500         2      2   
1       1.0       1    0  38.0      1      0  71.2833         0      3   
2       1.0       3    0  26.0      0      0   7.9250         2      1   
3       1.0       1    0  35.0      1      0  53.1000         2      3   
4       0.0       3    1  35.0      0      0   8.0500         2      2   
5       0.0       3    1  29.0      0      0   8.4583         1      

Epoch,▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇█████
Training loss,█▇▆▆▅▅▅▄▄▄▃▄▃▄▃▃▃▃▃▃▃▂▃▂▂▂▁▂▂▁▁▁▁▁▁▂▁▁▁▁
Validation accuracy,▁▄█▅▄▄▅▄▄▂▅▅▅▅▄▆▆▆▅▅▄▄▃▄▃▂▄▄▃▃▃▄▃▄▂▄▂▄▂▄
Validation loss,█▇▂▁▁▁▁▂▂▂▃▂▂▂▂▂▂▃▃▃▃▂▁▂▃▂▃▃▃▂▂▃▂▃▂▄▃▅▄▃
Best epoch,111
Best validation accuracy,0.8764
Epoch,1000
Training loss,0.32156


In [12]:
%run titanic_training.py -a relu -b 32 -l 1e-4 --no-early_stop

Namespace(wandb=True, batch_size=32, epochs=1000, learning_rate=0.0001, activation='relu', seed=42, normalize=True, early_stop=False, patience=100, submission=None)
{'epochs': 1000, 'batch_size': 32, 'learning_rate': 0.0001, 'activation': 'relu', 'n_hidden_unit_list': [20, 20], 'seed': 42, 'normalize': True, 'early_stop': False, 'patience': 100}
Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'title', 'family_num', 'alone'],
      dtype='object')
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  title  \
0       0.0       3    1  22.0      1      0   7.2500         2      2   
1       1.0       1    0  38.0      1      0  71.2833         0      3   
2       1.0       3    0  26.0      0      0   7.9250         2      1   
3       1.0       1    0  35.0      1      0  53.1000         2      3   
4       0.0       3    1  35.0      0      0   8.0500         2      2   
5       0.0       3    1  29.0      0      0   8.4583         1  

Epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇████
Training loss,█████▆▆▆▆▆▆▆▅▅▅▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▁▁
Validation accuracy,▁▁▄▅██▇███▇▇▇███████████████████████████
Validation loss,█▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▁▁
Best epoch,1000
Best validation accuracy,0.74157
Epoch,1000
Training loss,0.61199


| Learning rate | Epoch 100 Validation loss | Epoch 1000 Training loss | Epoch 1000 Validation loss | Epoch 1000 Validation accuracy | 비고 |
|---|---|---|---|---|---|
| 1e-2 | **0.3634** | **0.3216** | 0.3709 | 0.8483 | epoch 100 무렵 최저, 이후 과적합 |
| 1e-3 | 0.5875 | 0.4065 | **0.3612** | **0.8876** | 1000 epoch까지 꾸준히 개선 |
| 1e-4 | 0.6791 | 0.6120 | 0.5876 | 0.7416 | 학습 속도가 너무 느림 (미수렴) |

- **lr 1e-2에서는 과적합이 뚜렷하게 나타났다.** epoch 100 무렵 이미 validation loss 0.3634, accuracy 0.8876에 도달했지만, 이후 training loss는 0.40에서 0.32로 계속 내려가는 반면 validation loss는 0.37~0.38 사이에서 오히려 나빠지고 accuracy도 0.848까지 떨어졌다. 모델이 일반적인 패턴 대신 train 데이터 자체를 외우기 시작한 것이다.
- **lr 1e-4는 너무 느렸다.** 1000 epoch 후의 loss가 lr 1e-3의 epoch 100 수준에 불과하다.
- **lr 1e-3이 가장 균형 잡혀** 안정적으로 가장 낮은 validation loss에 도달했다.

아래는 lr 1e-2 실험의 wandb 그래프이다. training loss는 계속 내려가는데 validation loss는 epoch 100 무렵 최저를 찍은 뒤 조금씩 올라가는 과적합 양상을 볼 수 있다.

![lr 1e-2 과적합 그래프](https://raw.githubusercontent.com/audgk54587-dev/dl-homework/main/lr001_overfit.png)

### 4-4. 최종 선택

| 하이퍼파라미터 | 선택 | 근거 |
|---|---|---|
| Activation | ReLU | 가장 낮은 validation loss (Leaky ReLU와 사실상 동점, 더 단순한 쪽 선택) |
| Batch size | 32 | 같은 epoch에서 업데이트 횟수가 가장 많아 가장 잘 수렴 |
| Learning rate | 1e-3 | 1e-2는 과적합, 1e-4는 미수렴 |

## 5. [요구사항 3] Early Stopping과 submission.csv 생성

### 5-1. 어느 epoch 시점에 테스트해야 하는가

4-3의 lr 1e-2 실험에서 보았듯이, 훈련을 오래 할수록 training loss는 계속 줄지만 validation loss는 어느 시점부터 더 좋아지지 않거나 오히려 나빠진다. 따라서 **마지막 epoch의 모델이 아니라 validation loss가 가장 낮았던 epoch의 모델로 테스트해야 한다.** validation loss는 모델이 처음 보는 데이터에 대한 성능을 추정한 값이므로, 이것이 가장 낮은 시점의 모델이 test 데이터에서도 가장 잘할 가능성이 높다.

이를 위해 다음과 같이 구현했다.
- **`EarlyStopping` 클래스**: 매 epoch마다 validation loss가 최저 기록을 깨면 그 시점의 가중치를 `deepcopy`로 저장한다. **patience(100) epoch 동안 기록이 깨지지 않으면** 더 훈련해도 일반화 성능이 좋아지지 않는다고 보고 훈련을 멈춘다.
- **최적 가중치 복원**: 훈련이 끝나면 저장해 둔 best epoch의 가중치로 되돌린다(`load_state_dict`).
- **`make_submission()`**: 복원한 모델로 test 418명을 예측하고, 전처리 과정에서 삭제된 PassengerId는 원본 `test.csv`에서 가져와 `submission.csv`를 만든다. test 데이터로더는 PassengerId 순서와 맞아야 하므로 shuffle하지 않는다.

lr 1e-3은 1000 epoch에서도 아직 개선 중이었으므로, 최대 epoch를 5000으로 넉넉하게 주고 언제 멈출지는 Early Stopping이 판단하게 했다.

### 5-2. 최종 모델 훈련 (ReLU, batch 32, lr 1e-3, 최대 5000 epoch, patience 100)

In [13]:
%run titanic_training.py -a relu -b 32 -l 1e-3 -e 5000 -p 100 --submission submission.csv

Namespace(wandb=True, batch_size=32, epochs=5000, learning_rate=0.001, activation='relu', seed=42, normalize=True, early_stop=True, patience=100, submission='submission.csv')
{'epochs': 5000, 'batch_size': 32, 'learning_rate': 0.001, 'activation': 'relu', 'n_hidden_unit_list': [20, 20], 'seed': 42, 'normalize': True, 'early_stop': True, 'patience': 100}
Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'title', 'family_num', 'alone'],
      dtype='object')
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  title  \
0       0.0       3    1  22.0      1      0   7.2500         2      2   
1       1.0       1    0  38.0      1      0  71.2833         0      3   
2       1.0       3    0  26.0      0      0   7.9250         2      1   
3       1.0       1    0  35.0      1      0  53.1000         2      3   
4       0.0       3    1  35.0      0      0   8.0500         2      2   
5       0.0       3    1  29.0      0      0   8.4583    

Epoch,▁▁▁▁▁▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇███
Training loss,██▇▇▇▇▇▇▇▇▅▅▅▅▅▄▃▃▃▃▂▂▂▂▂▂▂▂▁▂▁▁▁▁▂▁▁▁▁▁
Validation accuracy,▁▁▁▂▂▃▃▄▅▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇████████████████
Validation loss,█▇▇▇▇▆▆▆▆▅▄▃▃▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Best epoch,1080
Best validation accuracy,0.88764
Epoch,1180
Training loss,0.40512


### 5-3. submission.csv 확인

In [14]:
submission_df = pd.read_csv("submission.csv")
print("shape:", submission_df.shape)
print("PassengerId 범위:", submission_df["PassengerId"].min(), "~", submission_df["PassengerId"].max())
print(submission_df["Survived"].value_counts())
submission_df.head(10)

shape: (418, 2)
PassengerId 범위: 892 ~ 1309
Survived
0    256
1    162
Name: count, dtype: int64


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


### 5-4. 결과와 고찰

| 항목 | 값 |
|---|---|
| Early stopping 시점 | epoch 1180 |
| Best epoch (테스트에 사용한 모델) | 1080 |
| Best validation loss | 0.3606 |
| Best validation accuracy | 0.8876 |
| 예측 생존 비율 (test 418명) | 0.3876 |

- epoch 1080에서 validation loss가 최저(0.3606)를 기록했고, 이후 100 epoch 동안 개선이 없어 epoch 1180에서 훈련을 멈췄다. 테스트에는 epoch 1180이 아니라 **epoch 1080의 가중치**를 사용했다.
- epoch 1000까지의 수치가 3-2(정규화 적용, ReLU, batch 32, lr 1e-3)와 소수점까지 동일하다. 시드 고정 덕분에 실험이 정확히 재현됨을 확인할 수 있다.
- 예측 생존 비율 38.8%는 train 데이터의 실제 생존 비율(약 38.4%)과 거의 같아, 모델이 한쪽으로 치우치지 않고 예측하고 있음을 알 수 있다.
- lr 1e-3에서는 1000 epoch 이후 validation loss가 0.3612에서 0.3606으로 아주 조금만 좋아졌다. 즉 이 설정에서는 epoch 1000 무렵 이미 거의 수렴했고, Early Stopping이 불필요한 추가 훈련을 자동으로 막아 주었다.

## 6. [요구사항 4] Kaggle 제출 및 등수 확인

| 항목 | 값 |
|---|---|
| 제출 파일 | submission.csv (ReLU, batch 32, lr 1e-3, early stopping) |
| Public Score | **0.77990** |
| 순위 | 3400위 (2026-10-03 기준) |
| 참고: Validation accuracy | 0.8876 |

**제출 결과**

![Kaggle 제출 점수](https://raw.githubusercontent.com/audgk54587-dev/dl-homework/main/kaggle_score.png)

**Leaderboard 순위**

![Kaggle Leaderboard 순위](https://raw.githubusercontent.com/audgk54587-dev/dl-homework/main/kaggle_rank.png)

**고찰: validation accuracy(0.8876)와 Kaggle 점수(0.7799)의 차이**
- 약 11%p 차이가 났다. validation 데이터는 178명뿐이라 어떤 승객이 validation에 들어가느냐(분할 운)에 따라 정확도가 크게 달라질 수 있다.
- 또한 하이퍼파라미터와 best epoch를 모두 같은 validation 데이터를 기준으로 골랐기 때문에, 결과적으로 모델이 이 validation 데이터에 맞춰진 셈이 되어 validation 성능이 실제 성능을 과대평가하게 된다.
- 개선 방향: k-fold 교차 검증으로 여러 분할의 평균 성능을 보면 더 믿을 만한 추정이 가능하다. 또 best epoch를 찾은 뒤 train과 validation을 합친 891명 전체로 다시 훈련하면 학습 데이터가 늘어나 test 성능이 좋아질 수 있다.

## 7. 종합 고찰: 코딩을 통해 취득한 기술적 사항

1. **데이터 전처리의 중요성**: 같은 모델, 같은 하이퍼파라미터에서 feature 표준화 하나만으로 validation accuracy가 0.80에서 0.89로 올랐다. 모델 구조나 하이퍼파라미터를 바꾸기 전에 입력 데이터의 스케일부터 확인해야 한다는 것을 배웠다.
2. **회귀와 분류의 차이**: 원본 코드(회귀)를 분류 문제로 바꾸면서 출력 개수(1→2), 손실 함수(MSELoss→CrossEntropyLoss), target 자료형(FloatTensor→LongTensor), 평가 지표(loss→accuracy 추가)를 함께 바꿔야 했다. CrossEntropyLoss는 내부에서 softmax를 계산하므로 모델 마지막에 softmax를 넣지 않는다는 점도 확인했다.
3. **공정한 실험 설계**: 시드를 고정하고 한 번에 하나의 하이퍼파라미터만 바꿔야 결과 차이의 원인을 설명할 수 있었다. 또한 batch size 비교처럼 "같은 epoch"이라는 조건이 실제로는 업데이트 횟수가 다른 불공정한 비교가 될 수 있다는 점을 알게 되었다.
4. **과적합과 Early Stopping**: lr 1e-2 실험에서 training loss와 validation loss가 갈라지는 과적합을 직접 관찰했고, validation loss가 최저인 시점의 가중치를 저장했다가 복원하는 방식으로 이를 해결했다. 가중치를 저장할 때 `deepcopy`를 하지 않으면 이후 훈련으로 저장한 값이 함께 바뀐다는 점도 주의해야 했다.
5. **실험 추적 도구**: wandb의 `define_metric`으로 x축을 Epoch로 지정하고 최솟값/최댓값을 요약에 남기니, 여러 실험을 표와 그래프로 쉽게 비교할 수 있었다.
6. **validation 성능의 한계**: validation 데이터가 작으면 그 성능이 실제 test 성능을 과대평가할 수 있다는 것을 Kaggle 점수와의 차이로 확인했다.

## 숙제 후기

이번 과제를 통해 파이토치로 인공지능 모델을 직접 학습시켜 보고, 설정값을 조정하며 그 과정을 그래프로 확인할 수 있었다. 특히 설정을 어떻게 바꾸느냐에 따라 모델 성능이 바뀌는 걸 확인하며 최적화 작업의 중요성을 실감할 수 있었다.